# Time-dependent PDEs in FEniCSx

## Time is crucial for many processes

In [ ]:
from IPython.display import IFrame

IFrame(src="https://www.youtube.com/embed/CCmTY0PKGDs", width=560, height=315)

## The heat equation is the "Hello world" example of a time-dependent PDE

We will solve the simplest extension of the Poisson problem into
the time domain, the **heat equation**:

\begin{align*}
\frac{\partial u}{\partial t} + D u_{xx} &= f \qquad \text{ for } x \in \Omega \text{ and for } t \in [0, T], \\
u &= g \qquad \text{ for } x \in \partial \Omega \text{ and } t \in [0, T], \\
u &= u^0 \qquad \text{ for } x \in \Omega \text{ and } t = 0.
\end{align*}

Here, $D \in \mathbb R$. $\Omega$ denotes the spatial domain and $\partial \Omega$ the domain boundary. For example, for $\Omega = [-L, L]$, the boundary $\partial \Omega$ consists of the two points $-L$ and $L$. We start the simulation at time $t=0$ and run it until the final time $T$.


The solution $u = u(x, t)$, the right-hand side $f = f (x, t)$, and the
boundary value $g = g(x, t)$ may vary in space $(x)$
and time $(t)$. The initial value $u_0$ is a function of space only.

Let's see what a 2D solution of this problem looks like:

In [ ]:
from IPython.display import YouTubeVideo

# Replace "d1NK83vve6c" with your video's ID
YouTubeVideo("TvlIfSlLB0c", width=800, height=450)

In [ ]:
## An example solution in 2D
from IPython.display import HTML

HTML(
    '<iframe width="560" height="315" src="https://www.youtube.com/embed/TvlIfSlLB0c?si=PzzfnsV6ImNae_m5" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>'
)

**Question**: Which kind of initial/boundary conditions were used in the above simulation?

## Time-discretization of the heat equation

FEniCS cannot direct solve time-dependent equations. We therefore must approximate the time-derivative $\frac{\partial u}{\partial t}$; that is, we discretize in time. This will lead to a sequence of time-independent PDEs, which we *can* solve with FEniCS. 


Our goal is to compute the solution at a set of discrete time-levels $0 = t^0 < t^1 < ... < T^N = T$. 
The variables at the $n$-th time-level will be denoted with a superscript $n$:

$$ u^n \approx u(t^n) \\
f^n = f(t^n)
$$

We discretize in time using the **implicit Euler** method.

$$ \frac{\partial u}{\partial t} (t^n) \approx \frac{u^n - u^{n-1}}{\Delta t} $$
This leads to the semi-discretization of the heat equation ("semi"-discretization because we have discretized in time, but not yet in space):
$$
u^n - u^{n-1} - \Delta t D u^n_{xx} = \Delta t f_n 
$$


### Time stepping algorithm for the heat equation

1. Start with $u^0$ and choose a time step $\Delta t$ > 0.
2. For $n = 1, 2, ...$, solve for $u^n$: 
 
   $u^n − \Delta t D u^n_{xx} = u^{n-1} + \Delta t f^n$

## Variational problem for the heat equation

To obtain the fully-discretized system, we derive the variational formulation of the semi-discretized system:

Find $u^n, n=1, ..., N$ such that 
$$
 a(u^n, v) = L^n(v)  \quad \text{ for all } v
$$
where 
$$
  a(u^n, v) = \int_\Omega u^nv + D \Delta t u_x v_x \text{d}x\\
  L^n(v) = \int_\Omega u^{n-1}v + \Delta t f^nv \text{d}x
$$
Note that the bilinear form $a(u, v)$ is constant while the linear
form $L^n$ depends on $n$.

## Detailed time-stepping algorithm for the heat equation

The following skeleton shows how to solve a time-dependent problem in FEniCS:

* Define the mesh, functions spaces and Dirichlet boundary conditions
* Compute $u_0$ as the projection of the given initial value
* Define the forms $a$ and $L$
* Set $t=\Delta t$
* **while $t \leq T$ do**
    * Apply the boundary condition
    * Solve the $AU = L$ for $U$ and store in $u_1$
    * Set $t$ to $t + \Delta t$
    * Set $u_0 = u_1$ (get ready for next step)
* **end while**

## Some implementation tips

In [ ]:
from dolfinx import fem, mesh
from dolfinx.fem.petsc import (
    assemble_matrix,
    assemble_vector,
    apply_lifting,
    set_bc,
    create_vector,
)
from petsc4py import PETSc
from ufl import TestFunction, TrialFunction, dx, grad, inner
from mpi4py import MPI
import numpy as np

domain = mesh.create_unit_interval(MPI.COMM_WORLD, 10)
V = fem.functionspace(domain, ("CG", 1))

## Handling time-dependent expressions

We need to define a time-dependent expression for the boundary value:

In [ ]:
class BoundaryValue:
    def __init__(self, beta: float, t: float):
        self.beta = beta
        self.t = t

    def __call__(self, x):
        return 1 - self.beta * self.t + 0 * x[0]


beta = 1.2
t = 0.0
g = BoundaryValue(beta, t)

Updating parameter values:

In [ ]:
g.t = 2.0

## Interpolation

Now `g` is a custom class, which does not depend on the mesh. Our unknown will be a `Function`, which is something living on a particular mesh, so to set its initial value we need to interpolate g:

In [ ]:
# Initial condition
u_D = fem.Function(V)
u_D.interpolate(g)

print("g is a: ", type(g))
print("u_D is a: ", type(u_D))

## Implementing the variational problem

In [ ]:
# Time step
dt = 0.1

# Initial condition
u_D = fem.Function(V)
u_D.interpolate(g)

# Define a variable to store the solution at the previous time-step
u_n = fem.Function(V)

# Define a source term
f = fem.Constant(domain, 0.0)


# Define the Dirichlet boundary condition
def on_boundary(x):
    return np.isclose(x[0], 0) | np.isclose(x[0], 1)


boundary_facets = mesh.locate_entities_boundary(
    domain, domain.topology.dim - 1, on_boundary
)
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)

bc = fem.dirichletbc(u_D, boundary_dofs)

# Code the variational formulation
u = TrialFunction(V)
v = TestFunction(V)

a = u * v * dx + dt * inner(grad(u), grad(v)) * dx
L = u_n * v * dx + dt * f * v * dx

a = fem.form(a)
L = fem.form(L)

In [ ]:
A = assemble_matrix(a, bcs=[bc])
A.assemble()
b = create_vector(fem.extract_function_spaces(L))
uh = fem.Function(V)

In [ ]:
solver = PETSc.KSP().create(domain.comm)
solver.setOperators(A)
solver.setType(PETSc.KSP.Type.PREONLY)
pc = solver.getPC()
pc.setType(PETSc.PC.Type.LU)

## Implementing the time-stepping loop

In [ ]:
from matplotlib import pyplot as plt

t = 0
T = 1.0
g.t = t

x_coords = V.tabulate_dof_coordinates()[:, 0]
plt.figure(figsize=(8, 6))
plt.plot(x_coords, u_n.x.array, label=f"t={t:.1f} (IC)")


while t < T:
    # Update Diriclet boundary condition
    t += dt
    g.t += dt
    u_D.interpolate(g)

    # Update the right hand side reusing the initial vector
    with b.localForm() as loc_b:
        loc_b.set(0)
    assemble_vector(b, L)

    # Apply Dirichlet boundary condition to the vector
    apply_lifting(b, [a], [[bc]])
    b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, [bc])

    # Solve linear problem
    solver.solve(b, uh.x.petsc_vec)
    uh.x.scatter_forward()

    # Update solution at previous time step (u_n)
    u_n.x.array[:] = uh.x.array

    # Plot current step
    plt.plot(x_coords, uh.x.array, label=f"t={t:.2f}")

# Finalize Plot
plt.xlabel("x")
plt.ylabel("u")
plt.legend()
plt.title("Heat Equation")
plt.grid(True)
plt.show()

## The cable equations

The standard cable equation is a reaction-diffusion equation given by
$$
\frac{\partial u}{\partial t}  = \sigma u_{xx} + f(u, s)
$$
where $f(u, s)$ is a reaction term describing ionic fluxes across
the membrane.
* A linear $f(u)$ describes passive conductance through a leaky cable (dendrites).
* A cubic $f(u)$ gives the bistable equation with a propagating activation front.
* In general $f(u, s)$, where s is a vector describing the state of the cell membrane, typically governed by a system of ODEs.

## Exercise 1: The cable equation

Solve the linear, bistable cable equation on an interval $\Omega=[-L, L]$ in FEniCS 
\begin{align*}
\frac{\partial u}{\partial t} &= \sigma u_{xx} + f(u) \quad &&\text{ for } -L < x < L, \\
u_x &= 0 \quad &&\text{ for } x = -L \text{ and } x = L,
\end{align*}
with 
* $f(u) = Au$,
* $A = -0.1$,
* $\sigma = 1.0$,
* $L = 100$.

Implement an implicit Euler time-stepping scheme and solve the problem from $t=0$ to $T=250$ with a time step of $dt=2.5$. Use as initial condition

$$
u(x,0) = \frac{1}{2} \left( 1 - \tanh\left(\sqrt{\frac{-A}{8\sigma}} (x + 0.75L)\right) \right)
$$
Create a plot of the solution on every 10th time-step. What happens if you change the value and sign of A? To improve the performance of your solver, try making it assemble `A` only once.

## Exercise 2: Forward or backward?

Above, we discretized in time using the **implicit Euler** method, meaning that we replaced $ \frac{\partial u}{\partial t} (t^n)$ by $\frac{u^n - u^{n-1}}{\Delta t} $. This is also called the **backward** Euler method because we are approximating a derivative by a difference *backwards* in time.

We could also have used the **forward** or **explicit Euler** method by approximating $ \frac{\partial u}{\partial t} (t^n)$ by $\frac{u^{n+1} - u^n}{\Delta t} $. Note that now the difference is going *forwards* in time instead. Take the heat equation $$\frac {\partial u} {\partial t} = Du_{xx}$$ and discretize it using both methods. What does the resulting systems of equations look like? Why do you think one is called explicit and the other implicit?

Next, let us try to compare the two. Choose initial conditions and Dirichlet boundary conditions so that the exact solution becomes $u_e = \text{exp}(-t) \text{ sin}(x)$. Then, using a time step of $\Delta t$ = 0.1, derive the weak formulation and solve using first a backward Euler discretization, then a forward Euler discretization. Plot the computed solution alongside the exact solution, or compute the error. Which time discretization would you prefer?

## Exercise 3: 2D heat equation
(This exercise may be challenging without knowledge of vector calculus. Feel free to skip it.)

The heat equation can be extended to a 2D domain in the following way:
$$\frac{\partial u}{\partial t} - D \text{ div grad } u = f$$

Here, $\text{ grad }$ means the gradient (vector of $x$ and $y$ derivatives), and $\text{ div }$ means the divergence (sum of $x$ and $y$ derivatives).

To derive its weak formulation, you will need to generalize integration by parts to 2D. The relevant formula is:
$$\int_\Omega \psi \text{ div grad } \phi = - \int_\Omega \text{grad } \psi \cdot \text{ grad } \phi + \text{ boundary term}$$

Use this to derive the weak formulation of the 2D heat equation. Then solve the heat equation on a unit square with the Dirichlet boundary condition $u = x(1-x)$ on the entire boundary, and the initial condition $u=0$. The mesh can be generated as follows:

In [ ]:
domain = mesh.create_unit_square(MPI.COMM_WORLD, 10, 10)